In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,auc,confusion_matrix,roc_auc_score,f1_score

In [ ]:
x_train=pd.read_csv(r"C:\Projects\coustmer_churn\new data\x_train")
y_train=pd.read_csv(r"C:\Projects\coustmer_churn\new data\y_train")
x_test=pd.read_csv(r"C:\Projects\coustmer_churn\new data\x_test")
y_test=pd.read_csv(r"C:\Projects\coustmer_churn\new data\y_test")

In [ ]:
y_train['Chrun'].value_counts()

In [ ]:
weight={0:1,1:2.5}
m1=RandomForestClassifier(n_estimators=1000,max_depth=6,random_state=42,class_weight=weight)
m1.fit(x_train,y_train["Chrun"])

In [ ]:
y_hat=m1.predict(x_test)

In [ ]:
print(classification_report(y_pred=y_hat,y_true=y_test['Chrun']))

In [ ]:
import optuna
from xgboost import XGBClassifier

In [ ]:
from numpy import tril

weight={0:1,1:2.5}
def objective(trial):
    params={
        'tree_method': 'hist',
        'device': 'cuda',
        'eta':trial.suggest_float('eta',1e-3, 0.1, log=True),
        'max_depth':trial.suggest_int('max_depth',6,32),
        'subsample':trial.suggest_float("subsample",0.6,1.0),
        'n_estimators': trial.suggest_int('n_estimators', 500, 2500),
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 42,
        'gpu_platform_id': 0,
        'gpu_device_id': 0


    }
    m1=XGBClassifier(**params,class_weight=weight)
    m1.fit(x_train,y_train["Chrun"],eval_set=[(x_test,y_test['Chrun'])],verbose=False)
    y_hat = m1.predict(x_test)
    
    # Calculate F1 Score
    score = f1_score(y_test['Chrun'], y_hat)
    return score

In [ ]:
study = optuna.create_study(direction='maximize') 
study.optimize(objective, n_trials=100,show_progress_bar=True)
# print(f"\nBest XGBoost AUC: {study_xgb.best_value:.5f}")
# print("Best XGBoost params:", study_xgb.best_params)

In [ ]:
print(roc_auc_score(y_pred=y_hat,y_true=y_test['Chrun']))